# 01 — Introdução ao Qdrant

## Por que um banco de dados vetorial?

Você já sabe como gerar embeddings. Mas onde armazena 1 milhão deles e como faz buscas eficientes?

A resposta ingênua seria: numpy array + busca linear. Problema: busca linear em 1M vetores de 768d leva **segundos**. Para um sistema RAG em produção, você precisa de resultados em **milissegundos**.

**Bancos de dados vetoriais** como o Qdrant resolvem isso com índices especializados (HNSW) que permitem busca aproximada em O(log N) — tipicamente 5-20ms para milhões de vetores.

Mas o Qdrant faz mais que só armazenar vetores:
- **Payloads:** metadados JSON junto com cada vetor (título, data, categoria, URL...)
- **Filtros:** combina busca vetorial com filtros nos metadados em uma única query
- **CRUD completo:** criar, ler, atualizar, deletar pontos e coleções
- **Quantização nativa:** reduz memória em 4-32x com configuração simples

Este notebook cobre as operações fundamentais que você vai usar em todo projeto RAG.

> **Pré-requisito:** Qdrant rodando (`docker compose up -d` no diretório infrastructure/)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, Range
)
from sentence_transformers import SentenceTransformer
import uuid

# Conectar ao Qdrant
try:
    client = QdrantClient(host="localhost", port=6333)
    client.get_collections()
    print("Qdrant conectado")
except Exception as e:
    print(f"Erro: {e}")
    print("Certifique-se que o Qdrant está rodando: docker compose up -d")
    raise

model = SentenceTransformer("all-MiniLM-L6-v2")
COLLECTION = "artigos_tech"
DIMS = model.get_sentence_embedding_dimension()
print(f"Modelo: {DIMS}d")

## 1.1 Criando uma Coleção

Uma **coleção** no Qdrant é análoga a uma tabela num banco relacional — é onde você armazena vetores com as mesmas características (dimensão, métrica de distância).

Parâmetros fundamentais:
- **`size`**: dimensão dos vetores (deve ser igual à saída do modelo)
- **`distance`**: métrica de distância — use `COSINE` para texto na maioria dos casos

**Por que a distância é configurada na coleção?** O Qdrant otimiza o índice HNSW para uma métrica específica. Mudar depois exigiria reindexar tudo.

In [ ]:
# Recriar collection (limpa se já existia)
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
    print(f"Coleção '{COLLECTION}' anterior removida")

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=DIMS, distance=Distance.COSINE)
)
print(f"Coleção '{COLLECTION}' criada ({DIMS}d, COSINE)")
info = client.get_collection(COLLECTION)
print(f"Status: {info.status}, pontos: {info.points_count}")

## 1.2 Inserindo Documentos (Upsert)

Cada documento no Qdrant é chamado de **Point** e tem três componentes:
- **`id`**: identificador único (int ou UUID)
- **`vector`**: o embedding do documento
- **`payload`**: metadados em JSON livre — qualquer coisa que você queira filtrar depois

O **payload** é uma das features mais poderosas do Qdrant.
Diferente de bancos vetoriais simples, você pode armazenar informações ricas e depois filtrar por elas sem sair do banco.

`upsert` (update + insert): se o ID já existir, atualiza. Se não existir, cria. Ideal para reindexação incremental.

In [ ]:
# Documentos com metadados ricos
artigos = [
    {"id": 1, "texto": "HNSW é um algoritmo de indexação vetorial hierárquico para busca aproximada eficiente",
     "categoria": "algoritmos", "ano": 2020, "score_qualidade": 9.5},
    {"id": 2, "texto": "Qdrant é um banco de dados vetorial open-source escrito em Rust com suporte a payload filtering",
     "categoria": "ferramentas", "ano": 2021, "score_qualidade": 9.0},
    {"id": 3, "texto": "Embeddings densos representam o significado semântico de textos como vetores de alta dimensão",
     "categoria": "conceitos", "ano": 2019, "score_qualidade": 8.5},
    {"id": 4, "texto": "RAG combina retrieval de documentos com geração de texto para responder perguntas",
     "categoria": "arquiteturas", "ano": 2020, "score_qualidade": 9.2},
    {"id": 5, "texto": "Quantização escalar reduz embeddings de float32 para int8, economizando 75% de memória",
     "categoria": "otimização", "ano": 2022, "score_qualidade": 8.8},
    {"id": 6, "texto": "BM25 é o algoritmo clássico de busca textual baseado em frequência de termos",
     "categoria": "algoritmos", "ano": 1994, "score_qualidade": 7.5},
    {"id": 7, "texto": "Hybrid search combina dense retrieval com BM25 para melhor cobertura semântica e keyword",
     "categoria": "arquiteturas", "ano": 2023, "score_qualidade": 9.8},
]

# Gerar embeddings e upsert em batch
textos = [a["texto"] for a in artigos]
embeddings = model.encode(textos, normalize_embeddings=True)

points = [
    PointStruct(
        id=a["id"],
        vector=embeddings[i].tolist(),
        payload={"texto": a["texto"], "categoria": a["categoria"],
                 "ano": a["ano"], "score_qualidade": a["score_qualidade"]}
    )
    for i, a in enumerate(artigos)
]

client.upsert(collection_name=COLLECTION, points=points)
info = client.get_collection(COLLECTION)
print(f"Upsert concluído. Total de pontos: {info.points_count}")

## 1.3 Busca Semântica

A operação mais importante: dado um vetor de query, encontrar os K vetores mais similares.

O Qdrant usa HNSW internamente para fazer essa busca em O(log N).

Parâmetros:
- **`query_vector`**: embedding da query do usuário
- **`limit`**: quantos resultados retornar (top-K)
- **`score_threshold`**: mínimo de similaridade para incluir no resultado (opcional)

O resultado inclui o **score** de cada ponto — a similaridade cosine com a query.
Score > 0.8: muito relevante. Score < 0.3: provavelmente irrelevante.

In [ ]:
def buscar(query, limit=3, score_threshold=None):
    q_vec = model.encode(query, normalize_embeddings=True).tolist()
    resultados = client.search(
        collection_name=COLLECTION,
        query_vector=q_vec,
        limit=limit,
        score_threshold=score_threshold
    )
    return resultados

# Teste
query = "como funciona a indexação para busca vetorial eficiente"
print(f"Query: '{query}'\n")
for r in buscar(query, limit=3):
    print(f"Score {r.score:.3f} | {r.payload['texto'][:70]}...")
    print(f"         categoria={r.payload['categoria']}, ano={r.payload['ano']}")

### Interpretando os scores

Um score de 0.7+ indica alta relevância semântica. Note que o sistema encontra documentos
semanticamente similares à query mesmo sem compartilhar palavras exatas com ela —
"indexação para busca vetorial eficiente" encontra documentos sobre HNSW e Qdrant porque compartilham o *conceito*.

Isso é o poder fundamental dos embeddings para RAG.

## 1.4 Operações CRUD

O Qdrant suporta operações completas de CRUD sobre os pontos — útil para manter a base atualizada sem precisar reindexar tudo.

In [ ]:
# READ: buscar por ID
pontos = client.retrieve(collection_name=COLLECTION, ids=[1, 2], with_payload=True)
print("READ por ID:")
for p in pontos:
    print(f"  ID {p.id}: {p.payload['texto'][:60]}...")

# UPDATE: atualizar payload sem precisar re-embedar
client.set_payload(
    collection_name=COLLECTION,
    payload={"verificado": True, "revisor": "equipe_ML"},
    points=[1, 2]
)
ponto_atualizado = client.retrieve(collection_name=COLLECTION, ids=[1], with_payload=True)[0]
print(f"\nUPDATE — novo payload do ponto 1: verificado={ponto_atualizado.payload.get('verificado')}")

# DELETE: remover ponto específico
client.delete(collection_name=COLLECTION, points_selector=[6])
print(f"\nDELETE ponto 6 (BM25 antigo)")
print(f"Total após delete: {client.get_collection(COLLECTION).points_count} pontos")

## 1.5 Filtros por Payload: O Diferencial do Qdrant

Esta é a feature que separa o Qdrant de uma busca linear simples.

Você pode combinar busca vetorial com filtros nos metadados em **uma única operação eficiente**.
O Qdrant executa isso de forma otimizada — não é busca vetorial seguida de filtro (que seria lento).

**Estrutura de filtros:**
- `must`: AND — todos os filtros devem ser verdadeiros
- `should`: OR — pelo menos um deve ser verdadeiro
- `must_not`: NOT — nenhum desses deve ser verdadeiro

Isso permite queries complexas:
"artigos de algoritmos OU arquiteturas, publicados após 2019, com score de qualidade acima de 9.0"

In [ ]:
q_vec = model.encode("indexação e busca eficiente", normalize_embeddings=True).tolist()

# Filtro: só categoria "algoritmos" ou "arquiteturas", ano >= 2020
filtro = Filter(
    must=[
        FieldCondition(key="ano", range=Range(gte=2020))
    ],
    should=[
        FieldCondition(key="categoria", match=MatchValue(value="algoritmos")),
        FieldCondition(key="categoria", match=MatchValue(value="arquiteturas")),
    ]
)

resultados = client.search(
    collection_name=COLLECTION,
    query_vector=q_vec,
    query_filter=filtro,
    limit=3,
    with_payload=True
)

print("Busca semântica + filtro (algoritmos/arquiteturas, ano >= 2020):")
for r in resultados:
    print(f"  Score {r.score:.3f} | {r.payload['categoria']:12s} | {r.payload['ano']} | {r.payload['texto'][:55]}...")

### Por que filtros no banco vetorial são essenciais para RAG?

Imagine um sistema RAG para uma empresa com documentos de vários departamentos.
Quando um funcionário do RH faz uma pergunta, você não quer que o sistema retorne
documentos confidenciais de finanças — mesmo que sejam semanticamente similares.

Com filtros de payload:
```python
filtro = Filter(must=[FieldCondition(key="department", match=MatchValue(value="HR"))])
```

Outra aplicação comum: filtrar por data.
"Mostre apenas documentos dos últimos 6 meses" — adicione `created_at` ao payload e filtre na query.

In [ ]:
# Scroll: paginar todos os pontos sem query (útil para auditoria)
all_points, next_cursor = client.scroll(
    collection_name=COLLECTION, limit=10, with_payload=True
)
print(f"Scroll — {len(all_points)} pontos:")
for p in all_points:
    print(f"  ID {p.id}: {p.payload['categoria']:12s} | {p.payload['texto'][:55]}...")

## Resumo

| Operação | Método Qdrant | Uso típico |
|----------|--------------|-----------|
| Criar índice | `create_collection` | Uma vez, na inicialização |
| Indexar docs | `upsert` | Indexação inicial e incremental |
| Buscar | `search` | Cada query do usuário |
| Buscar + filtrar | `search` + `query_filter` | RAG multi-tenant, filtros temporais |
| Ler por ID | `retrieve` | Debug, auditoria |
| Deletar | `delete` | Remover docs desatualizados |
| Atualizar metadata | `set_payload` | Sem precisar re-embedar |
| Paginar todos | `scroll` | Exportação, auditoria, re-indexação |

**Próximos passos:**
- [02 — HNSW Indexing](02_hnsw_indexing.html): como o Qdrant faz buscas tão rápidas internamente?
- [03 — Quantization](03_quantization.html): como comprimir o índice para economizar memória